In [10]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import os
os.chdir("/home/username/D/Machiene Learning Books/Kaggle Exercises/titanic")
data = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [16]:
data.info()
data.describe()
data.Name.head()
data.Cabin.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 29 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PassengerId     891 non-null    int64  
 1   Survived        891 non-null    int64  
 2   Pclass          891 non-null    int64  
 3   Name            891 non-null    object 
 4   Sex             891 non-null    int64  
 5   Age             714 non-null    float64
 6   SibSp           891 non-null    int64  
 7   Parch           891 non-null    int64  
 8   Ticket          891 non-null    object 
 9   Fare            891 non-null    float64
 10  Cabin           204 non-null    object 
 11  Embarked        889 non-null    object 
 12  Title_Capt      891 non-null    bool   
 13  Title_Col       891 non-null    bool   
 14  Title_Countess  891 non-null    bool   
 15  Title_Don       891 non-null    bool   
 16  Title_Dr        891 non-null    bool   
 17  Title_Jonkheer  891 non-null    boo

0     NaN
1     C85
2     NaN
3    C123
4     NaN
Name: Cabin, dtype: object

In [12]:
data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)
data = pd.get_dummies(data, columns = ['Title'])
titleCols = [col for col in data.columns if col.startswith('Title_')]
y = data['Survived']
data['Sex'] = data['Sex'].map({'male' : 1, 'female' : 0})
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch'] + titleCols
X = data[features].copy()
X['Age'] = X['Age'].fillna(data.Age.median())
titanicModel = RandomForestClassifier()
titanicModel = titanicModel.fit(X, y)

In [13]:
test['Title'] = test['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)
test = pd.get_dummies(test, columns=['Title'])
test['Sex'] = test['Sex'].map({'male': 1, 'female': 0})

title_cols = [col for col in test.columns if col.startswith('Title_')]
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch'] + title_cols

testX = test[features].copy()
testX['Age'] = testX['Age'].fillna(testX['Age'].median())

testX = testX.reindex(columns=X.columns, fill_value=0)

predictions = titanicModel.predict(testX)

In [14]:
print (predictions)

[0 0 1 1 0 0 0 0 1 0 1 0 1 0 1 1 0 0 1 0 0 1 1 0 1 0 1 0 1 0 0 0 1 0 0 0 1
 0 0 1 0 1 0 1 1 0 0 1 1 1 1 0 1 1 0 0 0 0 0 1 0 0 0 1 1 1 1 0 0 1 0 0 1 0
 1 0 1 1 0 0 1 0 1 1 0 0 1 1 1 1 1 1 0 1 0 0 1 0 0 0 1 0 1 0 0 0 0 1 1 0 1
 1 1 1 1 0 1 1 1 1 1 0 1 0 1 0 0 1 1 0 0 1 0 0 0 0 0 0 1 0 0 1 0 0 1 0 1 0
 1 0 1 1 0 1 0 0 1 1 1 1 1 1 1 1 0 0 1 0 1 1 1 1 0 1 0 1 1 0 1 1 0 1 1 1 1
 0 1 0 0 0 0 1 1 0 1 0 1 1 0 1 1 1 1 1 0 1 0 0 1 0 0 1 0 0 0 1 1 1 1 1 1 0
 1 0 1 1 0 1 0 0 0 1 1 1 1 0 0 0 1 1 1 1 0 1 1 1 1 0 1 0 1 0 1 0 0 1 1 0 1
 0 0 0 1 1 0 1 1 1 1 0 0 1 1 1 1 1 0 0 0 0 1 1 1 0 1 0 1 1 1 1 1 0 1 1 0 0
 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 1 0 1 0 0 0 1 0 1
 0 1 0 0 0 0 1 0 0 0 1 1 1 0 1 0 1 1 0 0 0 1 1 1 1 1 0 0 1 1 1 1 0 0 1 1 0
 0 1 1 1 1 1 0 0 0 0 1 0 1 0 0 1 0 0 0 0 0 1 1 0 0 1 0 1 0 0 1 0 1 0 0 0 0
 0 1 1 1 1 1 1 1 0 1 1]


In [15]:
output = pd.DataFrame({'PassengerId': test.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index = False)